In [0]:
from pyspark.sql import SparkSession

# Initialize a SparkSession with Delta support
spark = SparkSession.builder \
    .appName("DeltaLakePractice") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

# Check if Spark is working
print("Spark Session Created Successfully!")

#### Createting Delta Table

In [0]:
# Sample DataFrame creation
data = [("Alice", 34), ("Bob", 45), ("Cathy", 29)]
columns = ["name", "age"]
df = spark.createDataFrame(data, columns)

# Save the DataFrame as a Delta table (overwrite mode replaces any existing data)
df.write.format("delta").mode("overwrite").save("/Volumes/cdc_tutorial/cdc_db/delta_lake_tutorial")

In [0]:
display(df)

##### Reading from delta table

In [0]:
delta_df = spark.read.format("delta").load("/Volumes/cdc_tutorial/cdc_db/delta_lake_tutorial")
display(delta_df)

In [0]:
new_data=[("David",40)]
new_df = spark.createDataFrame(new_data,columns)
new_df.write.format("delta").mode("append").save("/Volumes/cdc_tutorial/cdc_db/delta_lake_tutorial")
display(new_df)

In [0]:
display(delta_df)

In [0]:
# Updated data for overwrite
updated_data = [("Alice", 35), ("Bob", 46), ("Cathy", 30)]
updated_df = spark.createDataFrame(updated_data, columns)

# Overwrite the current contents of the Delta table
updated_df.write.format("delta").mode("overwrite").save("/Volumes/cdc_tutorial/cdc_db/delta_lake_tutorial")

In [0]:
display(updated_df)

In [0]:
display(delta_df)

In [0]:
# Querying an earlier version (version 0) of the Delta table:
historical_df = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load("/Volumes/cdc_tutorial/cdc_db/delta_lake_tutorial")
historical_df.show()

In [0]:
display(historical_df)

#### Schema Evaluation

In [0]:
# Create a DataFrame with an additional "country" column:
new_data = [("Alice", 34, "USA"), ("Bob", 45, "Canada")]
columns = ["name", "age", "country"]
new_df = spark.createDataFrame(new_data, columns)

# Append the new data to the Delta table with schema evolution enabled:
new_df.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save("/Volumes/cdc_tutorial/cdc_db/delta_lake_tutorial")

In [0]:
display(new_df)

In [0]:
display(delta_df)

In [0]:
display(historical_df)